In [1]:
# In the reference dataset preparation I follow Kosugi paper (link). Further text is taken from the publication:
# 1. Remove DELs and INSs from DGV dataset with low (< 1kb) or undefined length
# 2. removed DELs, DUPs, and INVs with ≧ 95% reciprocal overlap 
#    (≧ 90% reciprocal overlap for > 1 kb variants) in the DGV and long read/assembly data, resulting in the removal of 450 variants in total.
# 3. The merge of both the datasets was conducted by removing shorter ones of overlapped DELs 
#    with ≧ 70% reciprocal overlap, resulting in the inclusion of 1671 DELs, 979 INSs, 2611 DUPs, and 233 INVs specific to the DGV SV data.

In [ ]:
import os

workdir = '/vast/ai_projects/Papenfuss_SVEnsemble2/reference/'
os.chdir(workdir)
os.getcwd()

In [3]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import pyranges as pr

In [4]:
pd.set_option('display.max_columns', None)

In [5]:
dgv_path = 'data/NA12878.DGV.hg38.tsv'
long_path = 'data/NA12878.long.hg38.tsv'

In [6]:
dgv_path = '../downloads/GRCh38_hg38_variants_2020-02-25.txt'

In [6]:
# dgv_path = '../downloads/GRCh38_hg38_variants_2016-05-15.txt'

In [7]:
dgv = pd.read_csv(dgv_path, sep='\t', dtype={'chr': 'string'})

In [8]:
dgv = dgv.dropna(subset=['chr', 'samples'])
dgv = dgv[dgv.samples.str.contains('NA12878')]

In [9]:
dgv.loc[:, 'chr'] = 'chr' + dgv['chr']

In [10]:
dgv = dgv.rename(columns={'chr': 'Chromosome', 'start': 'Start', 'end': 'End'})

In [11]:
dgv.loc[:, 'variantsubtype'] = dgv.variantsubtype.str.replace('loss', 'deletion')

In [ ]:
dgv['variantsubtype'].value_counts()

In [13]:
# sum((dgv.length <= 1000) & (dgv.variantsubtype.str.match('insertion|deletion')))

In [14]:
# 1. Remove DELs and INSs from DGV dataset with low (< 1kb) or undefined length
dgv.loc[:, 'length'] = dgv.End - dgv.Start
short_insdel_idx = (dgv.length <= 1000) & (dgv.variantsubtype.str.match('insertion|deletion'))
dgv = dgv[~short_insdel_idx]

In [ ]:
sum(short_insdel_idx)

In [ ]:
dgv

In [ ]:
dgv['variantsubtype'].value_counts()

In [18]:
# 2. removed DELs, DUPs, and INVs with ≧ 95% reciprocal overlap 
#    (≧ 90% reciprocal overlap for > 1 kb variants) in the DGV and long read/assembly data, resulting in the removal of 450 variants in total.

dgv = pr.PyRanges(dgv)
dgv = dgv.join(dgv, report_overlap=True).df
dgv.loc[:, 'overlap_frac'] = dgv.Overlap * 2 / (dgv.length + dgv.length_b)
dgv.loc[:, 'overlapped'] = (((dgv.length + dgv.length_b) < 2000) & (dgv.overlap_frac > 0.95)) \
                               | (((dgv.length + dgv.length_b) > 2000) & (dgv.overlap_frac > 0.90))

dgv_for_rm_df = dgv[
    (dgv.variantaccession != dgv.variantaccession_b) \
    & dgv.overlapped \
    & dgv.variantaccession.isin(['deletion', 'duplication', 'inversion']) \
    & dgv.variantaccession == dgv.variantaccession_b
]
dgv_for_rm = (pd.concat([dgv_for_rm_df['variantaccession'], dgv_for_rm_df['variantaccession_b']])).unique()

dgv = dgv[~dgv['variantaccession'].isin(dgv_for_rm)]
dgv = dgv[['variantaccession', 'Chromosome', 'Start', 'End', 'variantsubtype', 'reference', 'method', 'samples', 'length']].drop_duplicates()

In [ ]:
dgv['variantsubtype'].value_counts()

In [20]:
long = pd.read_csv(long_path, sep='\t')
long = long.rename(columns={'Chrom': 'Chromosome'})
long.loc[:, 'length'] = long.End - long.Start

In [ ]:
long['Event Type'].value_counts()

In [22]:
long = long.reset_index()
long = pr.PyRanges(long)
long = long.join(long, report_overlap=True).df
long.loc[:, 'overlap_frac'] = long.Overlap * 2 / (long.length + long.length_b)
long.loc[:, 'overlapped'] = (((long.length + long.length_b) < 2000) & (long.overlap_frac > 0.95)) \
                               | (((long.length + long.length_b) > 2000) & (long.overlap_frac > 0.90))
long_for_rm_df = long[
    (long.index != long.index_b) \
    & long.overlapped \
    & long['Event Type'].isin(['deletion', 'duplication', 'inversion']) \
    & long['Event Type'] == long['Event Type_b']
]
long_for_rm = (pd.concat([long_for_rm_df['index'], long_for_rm_df['index_b']])).unique()

long = long[~long['index'].isin(long_for_rm)]
long = long[['index', 'Chromosome', 'Start', 'End', 'Event Type', 'length']].drop_duplicates()

In [23]:
dgv_pr = pr.PyRanges(dgv)
long_pr = pr.PyRanges(long)
joined_dgv_long = long_pr.join(dgv_pr, report_overlap=True)
joined_dgv_long = joined_dgv_long.df
joined_dgv_long.loc[:, 'overlap_frac'] = joined_dgv_long.Overlap * 2 / (joined_dgv_long.End - joined_dgv_long.Start + joined_dgv_long.End_b - joined_dgv_long.Start_b)

# IDs for vars with any overlaps will be used to exclude them from the other datasets before merging
# index_overlapping = joined_dgv_long['index'].unique()
# variantaccession_overlapping = joined_dgv_long['variantaccession'].unique()

dgv_shorter = joined_dgv_long.length_b < joined_dgv_long.length
long_rm_idx = ~dgv_shorter & (joined_dgv_long.overlap_frac > 0.70)
dgv_rm_idx = dgv_shorter & (joined_dgv_long.overlap_frac > 0.70)
long_rm = joined_dgv_long[~long_rm_idx]['index'].unique()
dgv_rm = joined_dgv_long[~dgv_rm_idx]['variantaccession'].unique()
# joined_dgv_long = joined_dgv_long[~long_rm & ~dgv_rm]

# dgv_nonoverlap = dgv[~dgv.variantaccession.isin(variantaccession_overlapping)]

# long_nonoverlap = long[~long['index'].isin(variantaccession_overlapping)]

# joined_dgv_long = pd.concat([joined_dgv_long, dgv_nonoverlap, long_nonoverlap])

In [24]:
dgv_rename_dict = {
    'variantsubtype': 'ALT'
}
long_rename_dict = {
    'Event Type': 'ALT'
}

In [25]:
dgv = dgv.rename(columns=dgv_rename_dict)
long = long.rename(columns=long_rename_dict)

In [ ]:
dgv_to_join = dgv[~dgv.variantaccession.isin(dgv_rm)]
dgv_to_join.loc[:, 'source'] = 'dgv'
dgv_to_join.loc[:, 'ID'] = dgv_to_join['variantaccession']
dgv_to_join = dgv_to_join.drop(columns=['variantaccession'])

In [ ]:
long_to_join = long[~long['index'].isin(long_rm)]
long_to_join.loc[:, 'source'] = 'long'
long_to_join.loc[:, 'ID'] = 'long' + long_to_join['index'].astype(str)
long_to_join = long_to_join.drop(columns=['index'])

In [ ]:
full = pd.concat([dgv_to_join, long_to_join])
full

In [78]:
# Finally, GIAB svclassify DELs and INSs are added

In [29]:
giab_dels_path = 'data/Personalis_1000_Genomes_deduplicated_deletions.bed'
giab_ins_path = 'data/Spiral_Genetics_insertions.bed'

In [30]:
giab_dels = pd.read_csv(giab_dels_path, sep='\t')
giab_ins = pd.read_csv(giab_ins_path, sep='\t')

In [31]:
giab_dels.loc[:, 'ALT'] = 'deletion'
giab_ins.loc[:, 'ALT'] = 'insertion'

In [ ]:
giab = pd.concat([giab_dels, giab_ins]).reset_index(drop=True)
giab = giab.rename(columns={'Chr': 'Chromosome'})
giab.loc[:, 'Chromosome'] = 'chr' + giab.Chromosome.astype(str)
giab.loc[:, 'source'] = 'giab'
giab.loc[:, 'ID'] = 'giab' + giab.index.astype(str)
giab.loc[:, 'length'] = giab.End - giab.Start
giab

In [33]:
giab_pr = pr.PyRanges(giab)
full_pr = pr.PyRanges(full)

In [ ]:
full_pr

In [ ]:
giab_pr_overlapping = giab_pr.join(full_pr)
giab_rm = giab_pr_overlapping.ID.unique()
giab = giab[~giab.ID.isin(giab_rm)]
giab

In [ ]:
full = pd.concat([full, giab])
full

In [ ]:
full['ALT'].value_counts()

In [178]:
na12878_path = 'saved/NA12878.reference.tsv'
full.to_csv(na12878_path, sep='\t')

In [7]:
# Alternative: save the file from the authors in the same format as the one I created myself:
ready_vcf_path = 'data/NA12878_DGV-2016_LR-assembly.vcf'

In [96]:
ready_vcf_df = pd.read_csv(ready_vcf_path, sep='\t', names=['CHROM', 'POS', 'ALT', 'X1', 'X2', 'X3', 'FILTER', 'INFO'])

In [97]:
ready_vcf_df[['SVTYPE', 'SVLEN', 'SOURCE']] = ready_vcf_df.pop('INFO').str.split(';', expand=True)

In [98]:
ready_vcf_df.SVTYPE = ready_vcf_df.SVTYPE.str.split('=', expand=True).loc[:, 1]
ready_vcf_df.SVLEN = ready_vcf_df.SVLEN.str.split('=', expand=True).loc[:, 1].astype(int)
ready_vcf_df.SOURCE = ready_vcf_df.SOURCE.str.split('=', expand=True).loc[:, 1]

In [99]:
ready_vcf_df.loc[:, 'INFO_END'] = ready_vcf_df.POS + ready_vcf_df.SVLEN - 1
ready_vcf_df.CHROM = 'chr' + ready_vcf_df.CHROM

In [100]:
ready_vcf_df = ready_vcf_df[['CHROM', 'POS', 'INFO_END', 'ALT']]
# The resulting df is made for hg19, liftOver is needed to convert it to hg38
ready_vcf_df.to_csv('data/NA12878_DGV-2016_LR-assembly.bed', sep='\t', header=False, index=False)

In [ ]:
# !module load ucsc-tools
!liftOver NA12878_DGV-2016_LR-assembly.bed ../../utils/liftover/hg19ToHg38.over.chain.gz NA12878_DGV-2016_LR-assembly.hg38.bed NA12878_DGV-2016_LR-assembly.19to38.unmapped

In [8]:
ready_vcf_df_38_path = 'data/NA12878_DGV-2016_LR-assembly.hg38.bed'
ready_vcf_df_38 = pd.read_csv(ready_vcf_df_38_path, sep='\t', names=['Chromosome', 'Start', 'End', 'ALT'])

In [ ]:
ready_vcf_df_38.value_counts('ALT')

In [ ]:
ready_vcf_df_38_pr = pr.PyRanges(ready_vcf_df_38)
# full_pr = pr.PyRanges(full)

In [105]:
local_vs_ready = ready_vcf_df_38_pr.join(full_pr, report_overlap=True).df

In [ ]:
local_vs_ready

In [108]:
local_vs_ready.loc[:, 'overlap_frac'] = (local_vs_ready.Overlap * 2 / (local_vs_ready.End - local_vs_ready.Start + local_vs_ready.End_b - local_vs_ready.Start_b))

In [ ]:
local_vs_ready.overlap_frac.describe()